In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/reference.csv
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/AIMO3_Reference_Problems.pdf
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/sample_submission.csv
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/test.csv
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/aimo_3_inference_server.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/aimo_3_gateway.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/__init__.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/core/templates.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/core/base_gateway.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/core/relay.py
/kaggle/input/comp

In [2]:
# Install required packages
!pip install torch_geometric wandb -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 72.7 MB/s eta 0:00:00


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler

import polars as pl
import numpy as np
from tqdm.notebook import tqdm
import random
import gc
import os
import pickle
import math
from collections import defaultdict

import wandb
from sklearn.model_selection import train_test_split

In [4]:
# Configuration
SEED = 1023
TYPE_WEIGHTS = {'clicks': 1.0, 'carts': 6.0, 'orders': 3.0}

class Config:
    seed = SEED
    embed_dim = 64  # Increased from 32
    num_layers = 4   # Increased from 3
    batch_size = 4096
    epochs = 50
    learning_rate = 5e-4  # Slightly lower for stability
    weight_decay = 1e-5
    session_sample_rate = 1  # Increased from 5%
    validation_split = 0.1
    warmup_epochs = 3
    early_stopping_patience = 5
    
    # Edge sampling config
    sample_size = 15  # Neighbor sample size per layer

config = Config()

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(config.seed)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

Using device: cuda


In [5]:
# Data loading
type_map = {'clicks': 0, 'carts': 1, 'orders': 2}
type_weight_map = {'clicks': 1.0, 'carts': 6.0, 'orders': 3.0}

def load_raw_data_parquet(file_pattern):
    return pl.scan_parquet(file_pattern) \
        .with_columns(pl.col('type').replace(type_map).cast(pl.Int8)) \
        .with_columns((pl.col('ts') / 1000).cast(pl.Int32)) \
        .with_columns(pl.col('aid').cast(pl.Int32)) \
        .collect()

In [6]:
# Explore available datasets first
import glob
import os

print("Available input files:")
for root, dirs, files in os.walk('/kaggle/input'):
    level = root.replace('/kaggle/input', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:  # Only show 2 levels
        subindent = ' ' * 2 * (level + 1)
        for f in files[:5]:  # Show first 5 files
            print(f'{subindent}{f}')
        if len(files) > 5:
            print(f'{subindent}... and {len(files)-5} more')

# Find parquet files
parquet_files = glob.glob('/kaggle/input/**/*.parquet', recursive=True)
parquet_files += glob.glob('/kaggle/input/**/*train*.parquet', recursive=True)
parquet_files = list(set(parquet_files))[:20]  # Unique, first 20
print(f"\nFound parquet files: {parquet_files}")

# Load data - try multiple approaches
try:
    # Try as single file first
    if parquet_files:
        sample_file = parquet_files[0]
        print(f"\nLoading from: {sample_file}")
        sample = pl.read_parquet(sample_file)
        print(f"Schema: {sample.columns} - {sample.dtypes}")
        
        # Check columns and load all
        if 'type' in sample.columns:
            if sample['type'].dtype == pl.String:
                full_train_df = pl.concat([
                    pl.read_parquet(f).with_columns(
                        pl.col('type').replace(type_map).cast(pl.Int8)
                    )
                    for f in parquet_files if 'train' in f
                ])
            else:
                full_train_df = pl.concat([
                    pl.read_parquet(f)
                    for f in parquet_files if 'train' in f
                ])
        else:
            full_train_df = pl.concat([pl.read_parquet(f) for f in parquet_files if 'train' in f])
        
        print(f"Full training data shape: {full_train_df.shape}")
    else:
        raise FileNotFoundError("No parquet files found")
except Exception as e:
    print(f"Error: {e}")
    print("\nPlease check your dataset structure and update paths accordingly.")

Available input files:
input/
  competitions/
    ai-mathematical-olympiad-progress-prize-3/
      kaggle_evaluation/
        core/
          generated/
  datasets/
    cdeotte/
      otto-validation/
        test_parquet/
        train_parquet/

Found parquet files: ['/kaggle/input/datasets/cdeotte/otto-validation/train_parquet/085.parquet', '/kaggle/input/datasets/cdeotte/otto-validation/train_parquet/001.parquet', '/kaggle/input/datasets/cdeotte/otto-validation/train_parquet/016.parquet', '/kaggle/input/datasets/cdeotte/otto-validation/train_parquet/065.parquet', '/kaggle/input/datasets/cdeotte/otto-validation/train_parquet/057.parquet', '/kaggle/input/datasets/cdeotte/otto-validation/train_parquet/086.parquet', '/kaggle/input/datasets/cdeotte/otto-validation/train_parquet/018.parquet', '/kaggle/input/datasets/cdeotte/otto-validation/train_parquet/031.parquet', '/kaggle/input/datasets/cdeotte/otto-validation/train_parquet/070.parquet', '/kaggle/input/datasets/cdeotte/otto-validation

In [7]:
print(f"\n--- Step 1: Sampling {config.session_sample_rate*100}% of sessions ---")

all_train_sessions = full_train_df['session'].unique()
print(f"Total unique sessions: {len(all_train_sessions)}")

sampled_sessions = all_train_sessions.sample(
    fraction=config.session_sample_rate, 
    shuffle=True, 
    seed=config.seed
)
print(f"Sampled sessions: {len(sampled_sessions)}")

train_data = full_train_df.filter(pl.col('session').is_in(sampled_sessions))
print(f"Final training data shape: {train_data.shape}")

del full_train_df, all_train_sessions, sampled_sessions
gc.collect()


--- Step 1: Sampling 100% of sessions ---
Total unique sessions: 1775776
Sampled sessions: 1775776


/tmp/ipykernel_44/3163036712.py:13: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  train_data = full_train_df.filter(pl.col('session').is_in(sampled_sessions))


Final training data shape: (26798377, 4)


0

In [8]:
print("\n--- Creating Node Mappings ---")

unique_sessions = train_data['session'].unique()
unique_aids = train_data['aid'].unique()

num_sessions = len(unique_sessions)
num_aids = len(unique_aids)
total_nodes = num_sessions + num_aids

print(f"Unique sessions: {num_sessions}")
print(f"Unique AIDs: {num_aids}")
print(f"Total nodes: {total_nodes}")

session2idx = {s: i for i, s in enumerate(unique_sessions)}
aid2idx = {a: i + num_sessions for i, a in enumerate(unique_aids)}
idx2session = {i: s for s, i in session2idx.items()}
idx2aid = {i: a for a, i in aid2idx.items()}


--- Creating Node Mappings ---
Unique sessions: 1775776
Unique AIDs: 1402012
Total nodes: 3177788


In [9]:
print("\n--- Building Weighted Edge List ---")

# Get unique (session, aid) pairs with type info for edge weights
interactions = train_data.select(['session', 'aid', 'type']).unique()
print(f"Unique interactions: {len(interactions)}")

# Map to indices
interactions = interactions.with_columns([
    pl.col('session').replace(session2idx),
    pl.col('aid').replace(aid2idx)
])

# Create edge index and weights
source_nodes = interactions['session'].to_numpy()
target_nodes = interactions['aid'].to_numpy()
edge_types = interactions['type'].to_numpy()

# Create weighted edges
weights = np.array([type_weight_map.get(t, 1.0) for t in ['clicks', 'carts', 'orders']])[edge_types]

# Bidirectional edges
edge_index = np.array([
    np.concatenate([source_nodes, target_nodes]),
    np.concatenate([target_nodes, source_nodes])
])
edge_weights = np.concatenate([weights, weights])

print(f"Total edges (bidirectional): {edge_index.shape[1]}")

del interactions
gc.collect()


--- Building Weighted Edge List ---
Unique interactions: 18786215
Total edges (bidirectional): 37572430


0

In [10]:
from torch_geometric.data import Data

print("\n--- Preparing Graph Data ---")

# Convert to torch tensors
edge_index_tensor = torch.LongTensor(edge_index)
edge_weight_tensor = torch.FloatTensor(edge_weights)

# Create PyG Data object
edge_attr = edge_weight_tensor.unsqueeze(-1)  # Shape: [num_edges, 1]

graph_data = Data(
    edge_index=edge_index_tensor,
    edge_attr=edge_attr,
    num_nodes=total_nodes
).to(DEVICE)

print(f"Graph data prepared: {graph_data}")


--- Preparing Graph Data ---
Graph data prepared: Data(edge_index=[2, 37572430], edge_attr=[37572430, 1], num_nodes=3177788)


In [11]:
print("\n--- Preparing Training/Validation Split ---")

# Build user positive items dictionary
train_df_indexed = train_data.with_columns([
    pl.col('session').replace(session2idx),
    pl.col('aid').replace(aid2idx)
])

user_pos_items = train_df_indexed.group_by('session').agg(pl.col('aid'))
user_pos_dict = {row[0]: set(row[1]) for row in user_pos_items.rows()}

# Create interaction pairs
interaction_pairs = list(train_df_indexed.select(['session', 'aid']).unique().rows())
print(f"Total interaction pairs: {len(interaction_pairs)}")

# Train/Val split
train_pairs, val_pairs = train_test_split(
    interaction_pairs,
    test_size=config.validation_split,
    random_state=config.seed
)
print(f"Train pairs: {len(train_pairs)}, Val pairs: {len(val_pairs)}")

all_item_indices = set(aid2idx.values())

del train_data, train_df_indexed, user_pos_items
gc.collect()


--- Preparing Training/Validation Split ---
Total interaction pairs: 16670107
Train pairs: 15003096, Val pairs: 1667011


0

In [12]:
class BPRDataset(Dataset):
    def __init__(self, interaction_pairs, all_item_indices, user_pos_items_dict, num_items):
        self.interaction_pairs = interaction_pairs
        self.all_items = list(all_item_indices)
        self.user_pos_items = user_pos_items_dict
        self.num_items = num_items

    def __len__(self):
        return len(self.interaction_pairs)

    def __getitem__(self, idx):
        user_idx, pos_item_idx = self.interaction_pairs[idx]
        
        # Smart negative sampling
        neg_item_idx = random.choice(self.all_items)
        while neg_item_idx in self.user_pos_items.get(user_idx, set()):
            neg_item_idx = random.choice(self.all_items)
        
        return (
            torch.tensor(user_idx, dtype=torch.long),
            torch.tensor(pos_item_idx, dtype=torch.long),
            torch.tensor(neg_item_idx, dtype=torch.long)
        )

train_dataset = BPRDataset(train_pairs, all_item_indices, user_pos_dict, num_aids)
val_dataset = BPRDataset(val_pairs, all_item_indices, user_pos_dict, num_aids)

train_loader = DataLoader(
    train_dataset, 
    batch_size=config.batch_size, 
    shuffle=True, 
    num_workers=4, 
    pin_memory=True
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=config.batch_size, 
    shuffle=False, 
    num_workers=4, 
    pin_memory=True
)

print(f"DataLoaders prepared: Train={len(train_loader)}, Val={len(val_loader)}")

DataLoaders prepared: Train=3663, Val=407


In [13]:
from torch_geometric.nn.conv import GCNConv
from torch_geometric.nn.conv.gcn_conv import gcn_norm

class LightGCN(nn.Module):
    def __init__(self, num_users, num_items, embed_dim=64, num_layers=4, dropout=0.1):
        super().__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.embed_dim = embed_dim
        self.num_layers = num_layers
        self.dropout = nn.Dropout(dropout)
        
        self.embedding = nn.Embedding(num_users + num_items, embed_dim)
        nn.init.xavier_uniform_(self.embedding.weight)
        
        self.convs = nn.ModuleList([
            GCNConv(embed_dim, embed_dim, cached=False) 
            for _ in range(num_layers)
        ])
        
        self.agg_weights = nn.Parameter(torch.ones(num_layers + 1) / (num_layers + 1))

    def forward(self, edge_index, edge_weight=None):
        x0 = self.embedding.weight
        x = x0
        all_embeddings = [x0]
        
        # Pre-compute normalization
        norm_edge_index, norm_edge_weight = gcn_norm(
            edge_index, 
            num_nodes=self.num_users + self.num_items,
            edge_weight=edge_weight
        )
        
        for i, conv in enumerate(self.convs):
            x = conv.propagate(
                norm_edge_index, 
                x=x, 
                edge_weight=norm_edge_weight
            )
            x = self.dropout(x)
            all_embeddings.append(x)
        
        # Weighted aggregation
        weights = F.softmax(self.agg_weights, dim=0)
        final_emb = sum(w * emb for w, emb in zip(weights, all_embeddings))
        
        users_emb, items_emb = torch.split(
            final_emb, [self.num_users, self.num_items]
        )
        
        return users_emb, items_emb

model = LightGCN(
    num_users=num_sessions,
    num_items=num_aids,
    embed_dim=config.embed_dim,
    num_layers=config.num_layers
).to(DEVICE)

edge_index_tensor = edge_index_tensor.to(DEVICE)
edge_weight_tensor = edge_weight_tensor.to(DEVICE)

print(f"Model: {model}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

Model: LightGCN(
  (dropout): Dropout(p=0.1, inplace=False)
  (embedding): Embedding(3177788, 64)
  (convs): ModuleList(
    (0-3): 4 x GCNConv(64, 64)
  )
)
Total parameters: 203,395,077


In [14]:
# Optimizer and Scheduler
optimizer = optim.AdamW(
    model.parameters(),
    lr=config.learning_rate,
    weight_decay=config.weight_decay
)

def get_lr(epoch):
    if epoch < config.warmup_epochs:
        return (epoch + 1) / config.warmup_epochs
    else:
        progress = (epoch - config.warmup_epochs) / (config.epochs - config.warmup_epochs)
        return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=get_lr)
scaler = GradScaler()

/tmp/ipykernel_44/1008792385.py:16: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [15]:
def bpr_loss(users_emb, pos_emb, neg_emb):
    pos_scores = torch.sum(users_emb * pos_emb, dim=-1)
    neg_scores = torch.sum(users_emb * neg_emb, dim=-1)
    loss = -torch.mean(F.logsigmoid(pos_scores - neg_scores))
    return loss

def get_recall_at_k(users_emb, items_emb, user_pos_dict, k=20):
    recall_scores = []
    
    # Sample users for faster evaluation
    sample_size = min(1000, len(user_pos_dict))
    sampled_users = random.sample(list(user_pos_dict.keys()), sample_size)
    
    for user_idx in sampled_users:
        if user_idx >= len(users_emb):
            continue
            
        user_vec = users_emb[user_idx].unsqueeze(0)
        scores = torch.matmul(user_vec, items_emb.T).squeeze()
        
        _, top_k_indices = torch.topk(scores, min(k, len(scores)))
        top_k_items = set((top_k_indices + num_sessions).cpu().tolist())
        
        pos_items = user_pos_dict.get(user_idx, set())
        if len(pos_items) == 0:
            continue
            
        hits = len(top_k_items & pos_items)
        recall = hits / len(pos_items)
        recall_scores.append(recall)
    
    return np.mean(recall_scores) if recall_scores else 0.0

In [16]:
# Initialize WandB (optional - set to disabled if not using wandb)
try:
    wandb.init(
        project="otto-gnn-optimized",
        config={
            "embed_dim": config.embed_dim,
            "num_layers": config.num_layers,
            "batch_size": config.batch_size,
            "learning_rate": config.learning_rate,
            "session_sample_rate": config.session_sample_rate,
        },
        mode="disabled"  # Change to "online" if you have wandb account
    )
    USE_WANDB = True
except Exception as e:
    print(f"WandB not available: {e}")
    USE_WANDB = False


In [17]:
OUTPUT_DIR = "/kaggle/working/lightgcn_output/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

best_val_loss = float('inf')
best_recall = 0.0
patience_counter = 0

print("\n" + "="*60)
print("Starting Training")
print("="*60 + "\n")

for epoch in range(config.epochs):
    # Training Phase
    model.train()
    total_train_loss = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config.epochs} [Train]")
    for users, pos_items, neg_items in pbar:
        users = users.to(DEVICE)
        pos_items = (pos_items - num_sessions).to(DEVICE)  # Shift index
        neg_items = (neg_items - num_sessions).to(DEVICE)
        
        optimizer.zero_grad()
        
        with autocast():
            users_emb, items_emb = model(edge_index_tensor, edge_weight_tensor)
            
            user_emb_batch = users_emb[users]
            pos_emb_batch = items_emb[pos_items]
            neg_emb_batch = items_emb[neg_items]
            
            loss = bpr_loss(user_emb_batch, pos_emb_batch, neg_emb_batch)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_train_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    avg_train_loss = total_train_loss / len(train_loader)
    
    # Validation Phase
    model.eval()
    total_val_loss = 0
    
    with torch.no_grad():
        val_users_emb, val_items_emb = model(edge_index_tensor, edge_weight_tensor)
        
        for users, pos_items, neg_items in tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]"):
            users = users.to(DEVICE)
            pos_items = (pos_items - num_sessions).to(DEVICE)
            neg_items = (neg_items - num_sessions).to(DEVICE)
            
            user_emb_batch = val_users_emb[users]
            pos_emb_batch = val_items_emb[pos_items]
            neg_emb_batch = val_items_emb[neg_items]
            
            val_loss = bpr_loss(user_emb_batch, pos_emb_batch, neg_emb_batch)
            total_val_loss += val_loss.item()
        
        avg_val_loss = total_val_loss / len(val_loader)
        recall = get_recall_at_k(val_users_emb, val_items_emb, user_pos_dict, k=20)
    
    # Update scheduler
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    
    # Logging
    if USE_WANDB: wandb.log({
        'epoch': epoch + 1,
        'train_loss': avg_train_loss,
        'val_loss': avg_val_loss,
        'recall@20': recall,
        'learning_rate': current_lr
    })
    
    print(f"\nEpoch {epoch+1}/{config.epochs} - "
          f"Train Loss: {avg_train_loss:.4f}, "
          f"Val Loss: {avg_val_loss:.4f}, "
          f"Recall@20: {recall:.4f}, "
          f"LR: {current_lr:.6f}")
    
    # Early stopping based on recall
    if recall > best_recall:
        best_recall = recall
        patience_counter = 0
        best_model_path = os.path.join(OUTPUT_DIR, "lightgcn_best_recall.pth")
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'recall': recall,
            'val_loss': avg_val_loss,
        }, best_model_path)
        print(f"  ** New best model saved! Recall@20: {recall:.4f} **")
    else:
        patience_counter += 1
        
    if patience_counter >= config.early_stopping_patience:
        print(f"\nEarly stopping triggered after {epoch+1} epochs")
        break

print("\n" + "="*60)
print(f"Training Complete! Best Recall@20: {best_recall:.4f}")
print("="*60)


Starting Training



Epoch 1/50 [Train]:   0%|          | 0/3663 [00:00<?, ?it/s]

/tmp/ipykernel_44/335600975.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1 [Val]:   0%|          | 0/407 [00:00<?, ?it/s]


Epoch 1/50 - Train Loss: 0.4929, Val Loss: 0.3592, Recall@20: 0.0096, LR: 0.000333
  ** New best model saved! Recall@20: 0.0096 **


...

Epoch 12 [Val]:   0%|          | 0/407 [00:00<?, ?it/s]


Epoch 12/50 - Train Loss: 0.2191, Val Loss: 0.3260, Recall@20: 0.0107, LR: 0.000456


Epoch 13/50 [Train]:   0%|          | 0/3663 [00:00<?, ?it/s]

Epoch 13 [Val]:   0%|          | 0/407 [00:00<?, ?it/s]


Epoch 13/50 - Train Loss: 0.2143, Val Loss: 0.3287, Recall@20: 0.0110, LR: 0.000446

Early stopping triggered after 13 epochs

Training Complete! Best Recall@20: 0.0168


In [18]:
if USE_WANDB:
    wandb.finish()
